# 📘 Agentic 架构 8：情景记忆 + 语义记忆栈

欢迎阅读本系列第八个 notebook。今天讨论构建真正智能、长期助手的关键挑战之一：**持久记忆** 。标准对话记忆是短暂的，仅存在于单次会话。要打造随用户学习与成长的个性化智能体，需要更稳健的方案。

我们将实现一种模仿人类认知的结构化记忆架构，结合两类不同记忆：

1. **情景记忆（Episodic Memory）：** 对具体事件或过去交互的记忆，回答「发生了什么？」（例如「上周用户问过我 NVIDIA 股价」）。我们使用 **向量数据库** 检索与当前话题相关的历史对话。
2. **语义记忆（Semantic Memory）：** 从这些事件中抽取的结构化事实、概念与关系，回答「我知道什么？」（例如「用户 Alex 是保守型投资者」「Alex 关注科技股」）。我们使用 **图数据库（Neo4j）** ，因其擅长管理与查询复杂关系。

二者结合后，智能体不仅能回忆过去对话，还能构建关于用户与世界的丰富关联知识库，从而实现深度个性化与上下文感知交互。

### 定义
**情景 + 语义记忆栈** 是一种维护两类长期记忆的智能体架构。**情景记忆** 存储按时间排列的经历（如对话摘要），通常基于语义相似度检索。**语义记忆** 将抽取的结构化知识（实体、关系、事实）存入知识库，常为图结构。

### 高层工作流

1. **交互：** 智能体与用户对话。
2. **记忆检索（回忆）：** 对新查询，智能体先查询两套记忆系统。
    * 在 **情景** 向量库中检索相似历史对话。
    * 在 **语义** 图数据库中查询与问题相关的实体与事实。
3. **增强生成：** 将检索到的记忆加入提示上下文，使 LLM 能结合过往交互与已学事实生成回复。
4. **记忆创建（编码）：** 交互结束后，后台流程分析对话。
    * 生成本轮简洁摘要（新的 **情景** 记忆）。
    * 抽取关键实体与关系（新的 **语义** 记忆）。
5. **记忆存储：** 情景摘要嵌入后写入向量库；语义事实以节点与边写入图数据库。

### 适用场景 / 应用
* **长期个人助手：** 在数周或数月内记住偏好、项目与个人细节。
* **个性化系统：** 记住风格的电商机器人，或记住学习进度与薄弱点的辅导系统。
* **复杂研究智能体：** 在探索文档过程中构建主题知识图，以回答复杂多跳问题。

### 优势与局限
* **优势：**
    * **真正个性化：** 实现超越单次会话窗口、可长期持续的上下文与学习。
    * **丰富理解：** 图数据库使智能体能够理解与推理实体间的复杂关系。
* **局限：**
    * **复杂度：** 比无状态智能体明显更难构建与维护。
    * **记忆膨胀与剪枝：** 随时间推移存储可能急剧增长；摘要、合并或剪枝旧/无关记忆对长期性能至关重要。

## 阶段 0：基础与环境

安装全部必要库（含向量库与图数据库驱动），并配置 API 密钥。

In [ ]:
# !pip install -q -U langchain langgraph rich python-dotenv langchain_community langchain-openai neo4j faiss-cpu tiktoken

In [ ]:
import os
import uuid
from typing import List, Dict, Any, Optional, Tuple
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_nebius import NebiusEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv(overwriting=True)  # Load environment variables from .env file, allowing overwriting existing ones
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Memory Stack (DeepSeek + Nebius Embeddings)"

# Check for required environment variables
required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：构建记忆组件

这是架构核心：定义记忆结构、连接数据库，并创建负责处理对话、生成新记忆的「Memory Maker」智能体。

In [3]:
console = Console()
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0,
)
embeddings = NebiusEmbeddings()

# --- 1. Vector Store for Episodic Memory ---
# In a real application, you'd persist this. For this example, it's in-memory.
try:
    episodic_vector_store = FAISS.from_texts(["Initial document to bootstrap the store"], embeddings)
except ImportError:
    console.print("[bold red]FAISS not installed. Please run `pip install faiss-cpu`.")[/bold red]
    episodic_vector_store = None

# --- 2. Graph DB for Semantic Memory ---
try:
    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI"),
        username=os.environ.get("NEO4J_USERNAME"),
        password=os.environ.get("NEO4J_PASSWORD")
    )
    # Clear the graph for a clean run
    graph.query("MATCH (n) DETACH DELETE n")
except Exception as e:
    console.print(f"[bold red]Failed to connect to Neo4j: {e}. Please check your credentials and connection.[/bold red]")
    graph = None

# --- 3. Pydantic Models for the "Memory Maker" ---
# Define the structure of knowledge we want to extract.
class Node(BaseModel):
    id: str = Field(description="Unique identifier for the node, which can be a person's name, a company ticker, or a concept.")
    type: str = Field(description="The type of the node (e.g., 'User', 'Company', 'InvestmentPhilosophy').")
    properties: Dict[str, Any] = Field(description="A dictionary of properties for the node.")

class Relationship(BaseModel):
    source: Node = Field(description="The source node of the relationship.")
    target: Node = Field(description="The target node of the relationship.")
    type: str = Field(description="The type of the relationship (e.g., 'IS_A', 'INTERESTED_IN').")
    properties: Dict[str, Any] = Field(description="A dictionary of properties for the relationship.")

class KnowledgeGraph(BaseModel):
    """Represents the structured knowledge extracted from a conversation."""
    relationships: List[Relationship] = Field(description="A list of relationships to be added to the knowledge graph.")

# --- 4. The "Memory Maker" Agent ---
def create_memories(user_input: str, assistant_output: str):
    conversation = f"User: {user_input}\nAssistant: {assistant_output}"
    
    # 4a. Create Episodic Memory (Summarization)
    console.print("--- Creating Episodic Memory (Summary) ---")
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a summarization expert. Create a concise, one-sentence summary of the following user-assistant interaction. This summary will be used as a memory for future recall."),
        ("human", "Interaction:\n{interaction}")
    ])
    summarizer = summary_prompt | llm
    episodic_summary = summarizer.invoke({"interaction": conversation}).content
    
    new_doc = Document(page_content=episodic_summary, metadata={"created_at": uuid.uuid4().hex})
    episodic_vector_store.add_documents([new_doc])
    console.print(f"[green]Episodic memory created:[/green] '{episodic_summary}'")
    
    # 4b. Create Semantic Memory (Fact Extraction)
    console.print("--- Creating Semantic Memory (Graph) ---")
    extraction_llm = llm.with_structured_output(KnowledgeGraph)
    extraction_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a knowledge extraction expert. Your task is to identify key entities and their relationships from a conversation and model them as a graph. Focus on user preferences, goals, and stated facts."),
        ("human", "Extract all relationships from this interaction:\n{interaction}")
    ])
    extractor = extraction_prompt | extraction_llm
    try:
        kg_data = extractor.invoke({"interaction": conversation})
        if kg_data.relationships:
            for rel in kg_data.relationships:
                graph.add_graph_documents([rel], include_source=True)
            console.print(f"[green]Semantic memory created:[/green] Added {len(kg_data.relationships)} relationships to the graph.")
        else:
            console.print("[yellow]No new semantic memories identified in this interaction.[/yellow]")
    except Exception as e:
        console.print(f"[red]Could not extract or save semantic memory: {e}[/red]")

if episodic_vector_store and graph:
    print("Memory components initialized successfully.")

Memory components initialized successfully.


## 阶段 2：记忆增强型智能体

接下来构建使用该记忆系统的智能体。用 LangGraph 定义清晰的状态化工作流：检索记忆、用记忆生成回复、最后用最新交互更新记忆。

In [4]:
# Define the state for our LangGraph agent
class AgentState(TypedDict):
    user_input: str
    retrieved_memories: Optional[str]
    generation: str

# Define the nodes of the graph

def retrieve_memory(state: AgentState) -> Dict[str, Any]:
    """Node that retrieves memories from both episodic and semantic stores."""
    console.print("--- Retrieving Memories ---")
    user_input = state['user_input']
    
    # Retrieve from episodic memory
    retrieved_docs = episodic_vector_store.similarity_search(user_input, k=2)
    episodic_memories = "\n".join([doc.page_content for doc in retrieved_docs])
    
    # Retrieve from semantic memory
    # This is a simple retrieval; more advanced would involve entity extraction from the query
    try:
        graph_schema = graph.get_schema
        # Using a fulltext index for better retrieval. Neo4j automatically creates one on node properties.
        # A more robust solution might involve extracting entities from user_input first.
        semantic_memories = str(graph.query("""
            UNWIND $keywords AS keyword
            CALL db.index.fulltext.queryNodes("entity", keyword) YIELD node, score
            MATCH (node)-[r]-(related_node)
            RETURN node, r, related_node LIMIT 5
            """, {'keywords': user_input.split()}))
    except Exception as e:
        semantic_memories = f"Could not query graph: {e}"
        
    retrieved_content = f"Relevant Past Conversations (Episodic Memory):\n{episodic_memories}\n\nRelevant Facts (Semantic Memory):\n{semantic_memories}"
    console.print(f"[cyan]Retrieved Context:\n{retrieved_content}[/cyan]")
    
    return {"retrieved_memories": retrieved_content}

def generate_response(state: AgentState) -> Dict[str, Any]:
    """Node that generates a response using the retrieved memories."""
    console.print("--- Generating Response ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful and personalized financial assistant. Use the retrieved memories to inform your response and tailor it to the user. If the memories indicate a user's preference (e.g., they are a conservative investor), you MUST respect it."),
        ("human", "My question is: {user_input}\n\nHere are some memories that might be relevant:\n{retrieved_memories}")
    ])
    generator = prompt | llm
    generation = generator.invoke(state).content
    console.print(f"[green]Generated Response:\n{generation}[/green]")
    return {"generation": generation}

def update_memory(state: AgentState) -> Dict[str, Any]:
    """Node that updates the memory with the latest interaction."""
    console.print("--- Updating Memory ---")
    create_memories(state['user_input'], state['generation'])
    return {}

# Build the graph
workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve_memory)
workflow.add_node("generate", generate_response)
workflow.add_node("update", update_memory)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "update")
workflow.add_edge("update", END)

memory_agent = workflow.compile()
print("Memory-augmented agent graph compiled successfully.")

Memory-augmented agent graph compiled successfully.


## 阶段 3：演示与检查

观察智能体运行。我们模拟多轮对话：前两轮播种记忆，第三轮测试智能体是否能利用记忆做个性化回复；最后直接检查数据库中创建的记忆。

In [5]:
def run_interaction(query: str):
    result = memory_agent.invoke({"user_input": query})
    return result['generation']

console.print("\n--- 💬 INTERACTION 1: Seeding Memory ---")
run_interaction("Hi, my name is Alex. I'm a conservative investor, and I'm mainly interested in established tech companies.")

console.print("\n--- 💬 INTERACTION 2: Asking a specific question ---")
run_interaction("What do you think about Apple (AAPL)?")

console.print("\n--- 🧠 INTERACTION 3: THE MEMORY TEST ---")
run_interaction("Based on my goals, what's a good alternative to that stock?")


--- 💬 INTERACTION 1: Seeding Memory ---


--- Retrieving Memories ---
Retrieved Context:
Relevant Past Conversations (Episodic Memory):
Initial document to bootstrap the store

Relevant Facts (Semantic Memory):
[]
--- Generating Response ---
Generated Response:
Hello, Alex! It's great to meet you. As a conservative investor, focusing on established tech companies with strong fundamentals is a very sound strategy. I can certainly help you navigate that space. What's on your mind today?
--- Updating Memory ---
--- Creating Episodic Memory (Summary) ---
Episodic memory created: 'The user, Alex, introduced himself as a conservative investor interested in established tech companies.'
--- Creating Semantic Memory (Graph) ---
Semantic memory created: Added 2 relationships to the graph.



--- 💬 INTERACTION 2: Asking a specific question ---


--- Retrieving Memories ---
Retrieved Context:
Relevant Past Conversations (Episodic Memory):
The user, Alex, introduced himself as a conservative investor interested in established tech companies.
Initial document to bootstrap the store

Relevant Facts (Semantic Memory):
[]
--- Generating Response ---
Generated Response:
Apple (AAPL) is often considered a cornerstone for conservative tech portfolios. It has a massive market capitalization, a very strong brand, consistent profitability, and a history of returning value to shareholders through dividends and buybacks. Its ecosystem of products creates a loyal customer base, which provides a stable revenue stream. For a conservative investor, it generally aligns well with the goal of capital preservation while still offering growth potential. Would you like a deeper dive into its recent performance or financials?
--- Updating Memory ---
--- Creating Episodic Memory (Summary) ---
Episodic memory created: 'The user inquired about Apple (AAP


--- 🧠 INTERACTION 3: THE MEMORY TEST ---


--- Retrieving Memories ---
Retrieved Context:
Relevant Past Conversations (Episodic Memory):
The user, Alex, introduced himself as a conservative investor interested in established tech companies.
The user inquired about Apple (AAPL), and the assistant confirmed it's a suitable stock for conservative investors due to its stability and market position.

Relevant Facts (Semantic Memory):
[{'node': {'type': 'User', 'id': 'Alex', 'properties': {}}, 'r': {'type': 'HAS_GOAL', 'properties': {}}, 'related_node': {'type': 'InvestmentPhilosophy', 'id': 'Conservative Investing', 'properties': {}}}, {'node': {'type': 'User', 'id': 'Alex', 'properties': {}}, 'r': {'type': 'INTERESTED_IN', 'properties': {}}, 'related_node': {'type': 'Sector', 'id': 'Tech', 'properties': {}}}]
--- Generating Response ---
Generated Response:
Of course. Based on your stated goal of conservative investing in the tech sector, a great alternative to Apple (AAPL) would be Microsoft (MSFT).

Here's why it fits your profile

### 检查记忆存储

查看内部机制：可直接查询数据库，查看智能体创建的记忆。

In [6]:
console.print("--- 🔍 Inspecting Episodic Memory (Vector Store) ---")
# We'll do a similarity search for a general concept to see what comes up
retrieved_docs = episodic_vector_store.similarity_search("User's investment strategy", k=3)
for i, doc in enumerate(retrieved_docs):
    print(f"{i+1}. {doc.page_content}")

console.print("\n--- 🕸️ Inspecting Semantic Memory (Graph Database) ---")
print(f"Graph Schema:\n{graph.get_schema}")

# Cypher query to see who is interested in what
query_result = graph.query("MATCH (n:User)-[r:INTERESTED_IN|HAS_GOAL]->(m) RETURN n, r, m")
print(f"Relationships in Graph:\n{query_result}")

--- 🔍 Inspecting Episodic Memory (Vector Store) ---


1. The user, Alex, introduced himself as a conservative investor interested in established tech companies.
2. Based on the user's conservative investment goals, the assistant suggested Microsoft (MSFT) as a good alternative to Apple (AAPL), highlighting its diversification and enterprise focus.
3. The user inquired about Apple (AAPL), and the assistant confirmed it's a suitable stock for conservative investors due to its stability and market position.



--- 🕸️ Inspecting Semantic Memory (Graph Database) ---


Graph Schema:
{'node_props': {'InvestmentPhilosophy': [{'property': 'id', 'type': 'STRING'}], 'Company': [{'property': 'id', 'type': 'STRING'}], 'Sector': [{'property': 'id', 'type': 'STRING'}], 'User': [{'property': 'id', 'type': 'STRING'}]}, 'rel_props': {}, 'relationships': [{'start': 'User', 'type': 'INTERESTED_IN', 'end': 'Sector'}, {'start': 'User', 'type': 'INTERESTED_IN', 'end': 'Company'}, {'start': 'User', 'type': 'HAS_GOAL', 'end': 'InvestmentPhilosophy'}]}

Relationships in Graph:
[{'n': {'id': 'Alex', 'type': 'User'}, 'r': {}, 'm': {'id': 'Conservative Investing', 'type': 'InvestmentPhilosophy'}}, {'n': {'id': 'Alex', 'type': 'User'}, 'r': {}, 'm': {'id': 'Tech', 'type': 'Sector'}}, {'n': {'id': 'Alex', 'type': 'User'}, 'r': {}, 'm': {'id': 'AAPL', 'type': 'Company'}}, {'n': {'id': 'Alex', 'type': 'User'}, 'r': {}, 'm': {'id': 'MSFT', 'type': 'Company'}}]


## 结语

本 notebook 成功构建了带有复杂长期记忆系统的智能体。演示清晰展示了该架构的威力：

- **无状态失败：**标准智能体在问「根据我的目标，有什么好替代标的？」时会失败，因为它不记得用户目标。
- **记忆增强成功：**我们的智能体成功是因为能够：
    1. **情景回忆：**检索第一轮对话摘要：「用户 Alex 介绍自己是保守投资者……」
    2. **语义回忆：**查询图并得到结构化事实：`(User: Alex) -[HAS_GOAL]-> (InvestmentPhilosophy: Conservative)`。
    3. **综合：**结合上述上下文给出高度相关、个性化的建议（如微软），并明确引用用户的保守目标。

结合回忆 *发生了什么*（情景）与 *已知什么*（语义），是超越简单交易型智能体、走向真正学习型伙伴的有力范式。大规模管理记忆面临剪枝与合并等挑战，但此处搭建的基础架构是迈向更智能、更个性化 AI 系统的重要一步。